In [137]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/psrc0514/wifi-measurements/wifi_measurements.csv
/kaggle/input/datasets/psrc0514/survey-location/survey_location_profile.csv


In [138]:
import pandas as pd
import numpy as np

# -----------------------------
# Load Wi-Fi measurements
# -----------------------------

wifi_path = "/kaggle/input/datasets/psrc0514/wifi-measurements/wifi_measurements.csv"

df = pd.read_csv(wifi_path)

print("Wi-Fi dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Wi-Fi dataset shape: (429, 12)

Columns:
['Session', 'Location', 'Crowd Density', 'Time', 'BSSID', 'Signal (%)', 'Signal (dBm)', 'Ping (ms)', 'Link (Mbps)', 'DL (Mbps)', 'UL (Mbps)', 'Status']


,Session,Location,Crowd Density,Time,BSSID,Signal (%),Signal (dBm),Ping (ms),Link (Mbps),DL (Mbps),UL (Mbps),Status
0,NIGHT,DISANG ROAD,1,0:25:24,Unknown,92,-54,525,72.2,0.0,0.0,DEGRADED
1,NIGHT,DISANG ROAD,1,0:25:30,Unknown,92,-54,152,72.2,0.0,0.0,DEGRADED
2,NIGHT,DISANG ROAD,1,0:25:36,Unknown,92,-54,553,72.2,0.0,0.0,DEGRADED
3,NIGHT,DISANG ROAD,1,0:25:42,Unknown,92,-54,356,72.2,0.0,0.0,DEGRADED
4,NIGHT,DISANG ROAD,1,0:25:47,Unknown,92,-54,399,72.2,0.0,0.0,DEGRADED


In [139]:
# -----------------------------
# Load survey location profile
# -----------------------------

survey_path = "/kaggle/input/datasets/psrc0514/survey-location/survey_location_profile.csv"

survey_location_profile = pd.read_csv(survey_path)

print("Survey profile shape:", survey_location_profile.shape)

display(survey_location_profile)

Survey profile shape: (6, 2)


,location_group,survey_mentions
0,HOSTEL ROOM,49
1,ACADEMIC BUILDINGS,13
2,HOSTEL COMMON AREA,12
3,LIBRARY,7
4,MESS / CANTEEN,6
5,OTHER,3


In [140]:
print(df["Location"].unique())

print("\nSessions:")
print(df["Session"].unique())

print("\nShape:")
print(df.shape)

['DISANG ROAD' 'new sac' 'CORE5 GF' 'CLASSROOM' 'ROADS' 'CANTEEN'
 'VISITOR ROOM' 'ADMIN BUILDING' 'HOSTEL ROOM' 'LIBRARY' 'OPEN ROAD'
 'NEW SAC']

Sessions:
['NIGHT' 'MORNING']

Shape:
(429, 12)


In [141]:
print("Shape:", df.shape)

print("\nACTUAL COLUMNS:")
for i, col in enumerate(df.columns):
    print(i, repr(col))

print("\nFIRST 5 ROWS:")
display(df.head())

Shape: (429, 12)

ACTUAL COLUMNS:
0 'Session'
1 'Location'
2 'Crowd Density'
3 'Time'
4 'BSSID'
5 'Signal (%)'
6 'Signal (dBm)'
7 'Ping (ms)'
8 'Link (Mbps)'
9 'DL (Mbps)'
10 'UL (Mbps)'
11 'Status'

FIRST 5 ROWS:


,Session,Location,Crowd Density,Time,BSSID,Signal (%),Signal (dBm),Ping (ms),Link (Mbps),DL (Mbps),UL (Mbps),Status
0,NIGHT,DISANG ROAD,1,0:25:24,Unknown,92,-54,525,72.2,0.0,0.0,DEGRADED
1,NIGHT,DISANG ROAD,1,0:25:30,Unknown,92,-54,152,72.2,0.0,0.0,DEGRADED
2,NIGHT,DISANG ROAD,1,0:25:36,Unknown,92,-54,553,72.2,0.0,0.0,DEGRADED
3,NIGHT,DISANG ROAD,1,0:25:42,Unknown,92,-54,356,72.2,0.0,0.0,DEGRADED
4,NIGHT,DISANG ROAD,1,0:25:47,Unknown,92,-54,399,72.2,0.0,0.0,DEGRADED


In [142]:
# Make sure numeric columns are actually numeric

numeric_cols = [
    "Signal (dBm)",
    "Ping (ms)",
    "Link (Mbps)",
    "DL (Mbps)",
    "UL (Mbps)",
    "Crowd Density"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [143]:
network_profile = (
    df.groupby("Location")
      .agg(
          measurements=("Location", "size"),
          median_signal=("Signal (dBm)", "median"),
          median_download=("DL (Mbps)", "median"),
          ping_failure_rate=("Ping (ms)", "mean"),
          median_crowd=("Crowd Density", "median")
      )
      .reset_index()
)

network_profile["ping_failure_rate"] *= 100

display(network_profile)

,Location,measurements,median_signal,median_download,ping_failure_rate,median_crowd
0,ADMIN BUILDING,39,-60.0,40.46,44900.000000,1.0
1,CANTEEN,26,-81.0,0.34,89750.000000,2.0
2,CLASSROOM,65,-58.0,8.67,99900.000000,2.0
3,CORE5 GF,64,-59.0,9.07,99900.000000,5.0
4,DISANG ROAD,12,-54.0,4.78,33658.333333,1.0
5,HOSTEL ROOM,10,-55.0,5.19,99900.000000,3.0
6,LIBRARY,20,-63.5,0.34,99900.000000,3.0
7,NEW SAC,81,-58.0,0.00,75737.037037,4.0
8,OPEN ROAD,23,-83.0,0.00,99900.000000,1.0
9,ROADS,55,-53.0,62.03,18176.363636,1.0


In [144]:
survey_location_profile = pd.read_csv(
    "/kaggle/input/datasets/psrc0514/survey-location/survey_location_profile.csv"
)

display(survey_location_profile)

,location_group,survey_mentions
0,HOSTEL ROOM,49
1,ACADEMIC BUILDINGS,13
2,HOSTEL COMMON AREA,12
3,LIBRARY,7
4,MESS / CANTEEN,6
5,OTHER,3


In [145]:
survey_evidence = survey_location_profile.copy()

survey_evidence.columns = [
    "survey_category",
    "survey_mentions"
]

display(survey_evidence)

,survey_category,survey_mentions
0,HOSTEL ROOM,49
1,ACADEMIC BUILDINGS,13
2,HOSTEL COMMON AREA,12
3,LIBRARY,7
4,MESS / CANTEEN,6
5,OTHER,3


In [146]:
location_mapping = {
    "CANTEEN": "MESS / CANTEEN",
    "CLASSROOM": "ACADEMIC BUILDINGS",
    "HOSTEL ROOM": "HOSTEL ROOM",
    "LIBRARY": "LIBRARY"
}

network_profile["survey_category"] = (
    network_profile["Location"]
    .map(location_mapping)
)

display(network_profile)

,Location,measurements,median_signal,median_download,ping_failure_rate,median_crowd,survey_category
0,ADMIN BUILDING,39,-60.0,40.46,44900.000000,1.0,NaN
1,CANTEEN,26,-81.0,0.34,89750.000000,2.0,MESS / CANTEEN
2,CLASSROOM,65,-58.0,8.67,99900.000000,2.0,ACADEMIC BUILDINGS
3,CORE5 GF,64,-59.0,9.07,99900.000000,5.0,NaN
4,DISANG ROAD,12,-54.0,4.78,33658.333333,1.0,NaN
5,HOSTEL ROOM,10,-55.0,5.19,99900.000000,3.0,HOSTEL ROOM
6,LIBRARY,20,-63.5,0.34,99900.000000,3.0,LIBRARY
7,NEW SAC,81,-58.0,0.00,75737.037037,4.0,NaN
8,OPEN ROAD,23,-83.0,0.00,99900.000000,1.0,NaN
9,ROADS,55,-53.0,62.03,18176.363636,1.0,NaN


In [147]:
import pandas as pd
import numpy as np

# Reload original raw dataset
raw_df = pd.read_csv(
    "/kaggle/input/datasets/psrc0514/wifi-measurements/wifi_measurements.csv"
)

# -----------------------------
# Rename columns
# -----------------------------

df = raw_df.rename(columns={
    "Session": "session",
    "Location": "location",
    "Crowd Density": "crowd_density",
    "Time": "time",
    "BSSID": "bssid",
    "Signal (%)": "signal_percent",
    "Signal (dBm)": "signal_dbm",
    "Ping (ms)": "ping_ms",
    "Link (Mbps)": "link_mbps",
    "DL (Mbps)": "dl_mbps",
    "UL (Mbps)": "ul_mbps",
    "Status": "status"
}).copy()

# -----------------------------
# Clean categorical columns
# -----------------------------

df["location"] = (
    df["location"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df["session"] = (
    df["session"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# -----------------------------
# Clean numeric columns
# -----------------------------

numeric_cols = [
    "crowd_density",
    "signal_percent",
    "signal_dbm",
    "ping_ms",
    "link_mbps",
    "dl_mbps",
    "ul_mbps"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# -----------------------------
# Clean time
# -----------------------------

df["time"] = pd.to_datetime(
    df["time"].astype(str),
    format="%H:%M:%S",
    errors="coerce"
)

df["hour"] = df["time"].dt.hour
df["minute"] = df["time"].dt.minute

# -----------------------------
# Create ping failure indicator
# -----------------------------

df["ping_failed"] = (
    df["ping_ms"].isna() |
    (df["ping_ms"] <= 0)
).astype(int)

# -----------------------------
# Create poor connectivity target
# -----------------------------

DOWNLOAD_THRESHOLD = 1.0

df["poor_connectivity"] = (
    (df["dl_mbps"] <= DOWNLOAD_THRESHOLD) |
    (df["ping_failed"] == 1)
).astype(int)

print("Processed shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Processed shape: (429, 16)

Columns:
['session', 'location', 'crowd_density', 'time', 'bssid', 'signal_percent', 'signal_dbm', 'ping_ms', 'link_mbps', 'dl_mbps', 'ul_mbps', 'status', 'hour', 'minute', 'ping_failed', 'poor_connectivity']


,session,location,crowd_density,time,bssid,signal_percent,signal_dbm,ping_ms,link_mbps,dl_mbps,ul_mbps,status,hour,minute,ping_failed,poor_connectivity
0,NIGHT,DISANG ROAD,1,1900-01-01 00:25:24,Unknown,92,-54,525,72.2,0.0,0.0,DEGRADED,0,25,0,1
1,NIGHT,DISANG ROAD,1,1900-01-01 00:25:30,Unknown,92,-54,152,72.2,0.0,0.0,DEGRADED,0,25,0,1
2,NIGHT,DISANG ROAD,1,1900-01-01 00:25:36,Unknown,92,-54,553,72.2,0.0,0.0,DEGRADED,0,25,0,1
3,NIGHT,DISANG ROAD,1,1900-01-01 00:25:42,Unknown,92,-54,356,72.2,0.0,0.0,DEGRADED,0,25,0,1
4,NIGHT,DISANG ROAD,1,1900-01-01 00:25:47,Unknown,92,-54,399,72.2,0.0,0.0,DEGRADED,0,25,0,1


In [148]:
network_profile = (
    df.groupby("location")
      .agg(
          measurements=("location", "size"),
          median_signal=("signal_dbm", "median"),
          median_download=("dl_mbps", "median"),
          ping_failure_rate=("ping_failed", "mean"),
          median_crowd=("crowd_density", "median")
      )
      .reset_index()
)

network_profile["ping_failure_rate"] *= 100

display(network_profile)

,location,measurements,median_signal,median_download,ping_failure_rate,median_crowd
0,ADMIN BUILDING,39,-60.0,40.46,0.0,1.0
1,CANTEEN,26,-81.0,0.34,0.0,2.0
2,CLASSROOM,65,-58.0,8.67,0.0,2.0
3,CORE5 GF,64,-59.0,9.07,0.0,5.0
4,DISANG ROAD,12,-54.0,4.78,0.0,1.0
5,HOSTEL ROOM,10,-55.0,5.19,0.0,3.0
6,LIBRARY,20,-63.5,0.34,0.0,3.0
7,NEW SAC,101,-57.0,0.00,0.0,4.0
8,OPEN ROAD,23,-83.0,0.00,0.0,1.0
9,ROADS,55,-53.0,62.03,0.0,1.0


In [149]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

features = [
    "location",
    "session",
    "hour",
    "crowd_density",
    "signal_dbm"
]

target = "poor_connectivity"

X = df[features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            ["location", "session"]
        ),
        (
            "numeric",
            "passthrough",
            ["hour", "crowd_density", "signal_dbm"]
        )
    ]
)

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight="balanced"
            )
        )
    ]
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.7906976744186046

              precision    recall  f1-score   support

           0       0.87      0.77      0.82        53
           1       0.69      0.82      0.75        33

    accuracy                           0.79        86
   macro avg       0.78      0.80      0.78        86
weighted avg       0.80      0.79      0.79        86



In [150]:
from datetime import datetime
import re

def parse_time_input(time_input):

    if time_input is None:
        return None

    time_str = str(time_input).strip().upper()

    # Remove unnecessary spaces
    time_str = re.sub(r"\s+", " ", time_str)

    formats = [
        "%I:%M %p",   # 9:00 PM
        "%I %p",      # 9 PM
        "%I:%M%p",    # 9:00PM
        "%I%p",       # 9PM
        "%H:%M",      # 21:00
        "%H:%M:%S"    # 21:00:00
    ]

    for fmt in formats:
        try:
            return datetime.strptime(time_str, fmt).hour
        except ValueError:
            continue

    return None

In [151]:
def predict_wifi(location, time, crowd):

    location = location.upper().strip()

    parsed_time = pd.to_datetime(
        time,
        errors="coerce"
    )

    if pd.isna(parsed_time):
        return {
            "error": f"Could not understand time: {time}"
        }

    hour = parsed_time.hour

    if 5 <= hour < 12:
        session = "MORNING"
    elif 12 <= hour < 17:
        session = "AFTERNOON"
    else:
        session = "NIGHT"

    # Find historical measurements for this location/session
    historical = df[
        (df["location"].str.upper() == location) &
        (df["session"].str.upper() == session)
    ]

    if len(historical) == 0:

        # Fall back to location-level measurements
        historical = df[
            df["location"].str.upper() == location
        ]

    if len(historical) == 0:

        return {
            "error": f"No historical data available for {location}"
        }

    historical_signal = historical["signal_dbm"].median()

    input_data = pd.DataFrame({
        "location": [location],
        "session": [session],
        "hour": [hour],
        "crowd_density": [crowd],
        "signal_dbm": [historical_signal]
    })

    probability = rf_model.predict_proba(
        input_data
    )[0][1]

    prediction = int(probability >= 0.5)

    if probability >= 0.75:
        risk = "HIGH"
    elif probability >= 0.50:
        risk = "MEDIUM"
    else:
        risk = "LOW"

    return {
        "location": location,
        "time": time,
        "hour": hour,
        "session": session,
        "crowd": crowd,
        "historical_signal_dbm": historical_signal,
        "prediction": prediction,
        "poor_connectivity_probability": probability,
        "risk": risk
    }

In [152]:
def get_wifi_prediction(location, time, crowd):

    result = predict_wifi(
        location=location,
        time=time,
        crowd=crowd
    )

    if "error" in result:
        return result

    location = result["location"]

    # Historical network profile
    profile = network_profile[
        network_profile["location"].str.upper() == location
    ]

    if len(profile) > 0:

        profile = profile.iloc[0]

        result["median_download_mbps"] = profile["median_download"]
        result["ping_failure_rate"] = profile["ping_failure_rate"]
        result["median_crowd"] = profile["median_crowd"]
        result["survey_category"] = profile["survey_category"]
        result["survey_mentions"] = profile["survey_mentions"]

    else:

        result["median_download_mbps"] = None
        result["ping_failure_rate"] = None
        result["median_crowd"] = None
        result["survey_category"] = None
        result["survey_mentions"] = None

    return result

In [153]:
survey_location_profile = pd.read_csv(
    "/kaggle/input/datasets/psrc0514/survey-location/survey_location_profile.csv"
)

print(survey_location_profile.columns.tolist())
display(survey_location_profile)

['location_group', 'survey_mentions']


,location_group,survey_mentions
0,HOSTEL ROOM,49
1,ACADEMIC BUILDINGS,13
2,HOSTEL COMMON AREA,12
3,LIBRARY,7
4,MESS / CANTEEN,6
5,OTHER,3


In [154]:
survey_evidence.columns = [
    "survey_category",
    "survey_mentions"
]

survey_evidence["survey_category"] = (
    survey_evidence["survey_category"]
    .astype(str)
    .str.strip()
    .str.upper()
)

display(survey_evidence)

,survey_category,survey_mentions
0,HOSTEL ROOM,49
1,ACADEMIC BUILDINGS,13
2,HOSTEL COMMON AREA,12
3,LIBRARY,7
4,MESS / CANTEEN,6
5,OTHER,3


In [155]:
location_mapping = {
    "CANTEEN": "MESS / CANTEEN",
    "CLASSROOM": "ACADEMIC BUILDINGS",
    "HOSTEL ROOM": "HOSTEL ROOM",
    "LIBRARY": "LIBRARY"
}

In [156]:
network_profile["survey_category"] = (
    network_profile["location"]
    .map(location_mapping)
)

In [157]:
network_profile = network_profile.merge(
    survey_evidence,
    on="survey_category",
    how="left"
)

display(network_profile)

,location,measurements,median_signal,median_download,ping_failure_rate,median_crowd,survey_category,survey_mentions
0,ADMIN BUILDING,39,-60.0,40.46,0.0,1.0,NaN,NaN
1,CANTEEN,26,-81.0,0.34,0.0,2.0,MESS / CANTEEN,6.0
2,CLASSROOM,65,-58.0,8.67,0.0,2.0,ACADEMIC BUILDINGS,13.0
3,CORE5 GF,64,-59.0,9.07,0.0,5.0,NaN,NaN
4,DISANG ROAD,12,-54.0,4.78,0.0,1.0,NaN,NaN
5,HOSTEL ROOM,10,-55.0,5.19,0.0,3.0,HOSTEL ROOM,49.0
6,LIBRARY,20,-63.5,0.34,0.0,3.0,LIBRARY,7.0
7,NEW SAC,101,-57.0,0.00,0.0,4.0,NaN,NaN
8,OPEN ROAD,23,-83.0,0.00,0.0,1.0,NaN,NaN
9,ROADS,55,-53.0,62.03,0.0,1.0,NaN,NaN


In [158]:
result = get_wifi_prediction(
    location="LIBRARY",
    time="9:00 PM",
    crowd=3
)

print(result)

{'location': 'LIBRARY', 'time': '9:00 PM', 'hour': 21, 'session': 'NIGHT', 'crowd': 3, 'historical_signal_dbm': -63.5, 'prediction': 1, 'poor_connectivity_probability': np.float64(0.7207222080470177), 'risk': 'MEDIUM', 'median_download_mbps': np.float64(0.34), 'ping_failure_rate': np.float64(0.0), 'median_crowd': np.float64(3.0), 'survey_category': 'LIBRARY', 'survey_mentions': np.float64(7.0)}


In [159]:
def format_wifi_response(result):

    if "error" in result:
        return f" {result['error']}"

    location = result["location"]
    time = result["time"]
    crowd = result["crowd"]

    probability = result["poor_connectivity_probability"] * 100
    risk = result["risk"]

    download = result.get("median_download_mbps")
    ping_failure = result.get("ping_failure_rate")
    survey_mentions = result.get("survey_mentions")

    response = f"""
##  Campus Wi-Fi Forecast

 Location: {location}  
 Time: {time}  
 Expected Crowd: {crowd}

### Prediction

Risk: {risk}  
Probability of poor connectivity: {probability:.1f}%

###  Historical Network Evidence
"""

    if pd.notna(download):
        response += f"- Median download speed: **{download:.2f} Mbps**\n"

    if pd.notna(ping_failure):
        response += f"- Ping failure rate: **{ping_failure:.1f}%**\n"

    response += "\n###  Student Survey Evidence\n"

    if pd.notna(survey_mentions):
        response += (
            f"- **{int(survey_mentions)}** survey responses "
            f"mentioned this location/category.\n"
        )
    else:
        response += (
            "- No direct survey evidence is available for this location.\n"
        )

    response += "\n###  Recommendation\n"

    if risk == "HIGH":
        response += (
            " Connectivity is predicted to be poor. "
            "For important online work, consider downloading resources "
            "beforehand or using a more reliable location."
        )

    elif risk == "MEDIUM":
        response += (
            " Connectivity may be inconsistent. "
            "Keep a backup connection available for important work."
        )

    else:
        response += (
            "Connectivity risk is relatively low based on the "
            "available measurements."
        )

    return response

In [160]:
result = get_wifi_prediction(
    location="LIBRARY",
    time="9:00 PM",
    crowd=3
)

print(format_wifi_response(result))


##  Campus Wi-Fi Forecast

 Location: LIBRARY  
 Time: 9:00 PM  
 Expected Crowd: 3

### Prediction

Risk: MEDIUM  
Probability of poor connectivity: 72.1%

###  Historical Network Evidence
- Median download speed: **0.34 Mbps**
- Ping failure rate: **0.0%**

###  Student Survey Evidence
- **7** survey responses mentioned this location/category.

###  Recommendation
 Connectivity may be inconsistent. Keep a backup connection available for important work.


In [161]:
!pip -q install gradio

In [162]:
import gradio as gr

In [163]:
def chatbot_predict(location, time, crowd):

    result = get_wifi_prediction(
        location=location,
        time=time,
        crowd=int(crowd)
    )

    return format_wifi_response(result)

In [164]:
locations = sorted(
    df["location"].dropna().unique().tolist()
)

demo = gr.Interface(
    fn=chatbot_predict,
    inputs=[
        gr.Dropdown(
            choices=locations,
            label="Location"
        ),

        gr.Textbox(
            value="9:00 PM",
            label="Time"
        ),

        gr.Slider(
            minimum=1,
            maximum=5,
            step=1,
            value=3,
            label="Expected Crowd"
        )
    ],

    outputs=gr.Markdown(),

    title="Campus Wi-Fi Predictor",

    description=(
        "Predict Wi-Fi connectivity using historical measurements, "
        "crowd level, and student survey evidence."
    )
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7864
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://aa574bf9661b7d0fd9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [165]:
import re
import gradio as gr


# --------------------------------------------------
# Available locations from your actual dataset
# --------------------------------------------------

AVAILABLE_LOCATIONS = sorted(
    df["location"].dropna().astype(str).str.upper().unique()
)

print("Available locations:")
print(AVAILABLE_LOCATIONS)


# --------------------------------------------------
# Extract location
# --------------------------------------------------

def extract_location(text):

    text_upper = text.upper()

    # Try longest names first
    for location in sorted(AVAILABLE_LOCATIONS, key=len, reverse=True):

        if location in text_upper:
            return location

    return None


# --------------------------------------------------
# Extract time
# --------------------------------------------------

def extract_time(text):

    # Examples:
    # 9 PM
    # 9:00 PM
    # 10 AM
    # 21:00

    pattern_12 = r'\b(\d{1,2})(?::(\d{2}))?\s*(AM|PM)\b'
    match = re.search(pattern_12, text.upper())

    if match:

        hour = int(match.group(1))
        minute = int(match.group(2) or 0)
        meridian = match.group(3)

        if meridian == "PM" and hour != 12:
            hour += 12

        if meridian == "AM" and hour == 12:
            hour = 0

        return f"{hour:02d}:{minute:02d}"

    # 24-hour format
    pattern_24 = r'\b([01]?\d|2[0-3]):([0-5]\d)\b'

    match = re.search(pattern_24, text)

    if match:
        return match.group(0)

    return None


# --------------------------------------------------
# Extract crowd
# --------------------------------------------------

def extract_crowd(text):

    patterns = [
        r'crowd\s*(?:of|=|:)?\s*(\d+)',
        r'(\d+)\s*(?:people|students|persons)',
        r'(\d+)\s*students',
    ]

    text_lower = text.lower()

    for pattern in patterns:

        match = re.search(pattern, text_lower)

        if match:
            return int(match.group(1))

    return None


# --------------------------------------------------
# Chatbot
# --------------------------------------------------

def wifi_chatbot(message, history):

    location = extract_location(message)
    time = extract_time(message)
    crowd = extract_crowd(message)

    # ------------------------------------------
    # Missing information
    # ------------------------------------------

    missing = []

    if location is None:
        missing.append("location")

    if time is None:
        missing.append("time")

    if crowd is None:
        missing.append("crowd size")

    if missing:

        response = "I can predict the connectivity risk, but I need:\n\n"

        for item in missing:
            response += f"• {item}\n"

        response += "\nFor example:\n"
        response += "👉 `NEW SAC at 9 PM with crowd 4`"

        return response


    # ------------------------------------------
    # Run your existing prediction engine
    # ------------------------------------------

    result = get_wifi_prediction(
        location=location,
        time=time,
        crowd=crowd
    )

    if "error" in result:
        return result["error"]


    # ------------------------------------------
    # Extract results
    # ------------------------------------------

    probability = (
        result["poor_connectivity_probability"] * 100
    )

    risk = result["risk"]

    signal = result["historical_signal_dbm"]

    download = result["median_download_mbps"]

    ping_failure = result["ping_failure_rate"]

    survey_mentions = result.get("survey_mentions")

    survey_category = result.get("survey_category")


    # ------------------------------------------
    # Risk emoji
    # ------------------------------------------

    if risk == "HIGH":
        risk_icon = "🔴"

    elif risk == "MEDIUM":
        risk_icon = "🟠"

    else:
        risk_icon = "🟢"


    # ------------------------------------------
    # Build response
    # ------------------------------------------

    response = f"""
## 📡 Campus Wi-Fi Prediction

**Location:** {location}  
**Time:** {time}  
**Expected crowd:** {crowd}

### {risk_icon} Connectivity Risk: {risk}

**Estimated probability of poor connectivity:** `{probability:.1f}%`

### 📊 Network Evidence

- Median signal strength: **{signal:.1f} dBm**
- Median download speed: **{download:.2f} Mbps**
- Ping failure rate: **{ping_failure:.1f}%**
"""

    # ------------------------------------------
    # Survey evidence
    # ------------------------------------------

    if (
        survey_mentions is not None
        and pd.notna(survey_mentions)
        and survey_mentions > 0
    ):

        response += f"""

### 🧑‍🎓 Student Survey Evidence

The survey contains **{int(survey_mentions)} responses** associated with the **{survey_category}** category.
"""

    else:

        response += """

### 🧑‍🎓 Student Survey Evidence

No directly mapped student-survey category is currently available for this location.
"""


    # ------------------------------------------
    # Interpretation
    # ------------------------------------------

    response += "\n### 💡 Assessment\n"

    if risk == "HIGH":

        response += (
            "Based on the historical measurements and the supplied crowd "
            "condition, this location has a **high predicted risk of poor connectivity**."
        )

    elif risk == "MEDIUM":

        response += (
            "The supplied conditions correspond to a **moderate predicted "
            "connectivity risk** based on the collected measurements."
        )

    else:

        response += (
            "The supplied conditions correspond to a **lower predicted "
            "connectivity risk** based on the collected measurements."
        )


    return response

Available locations:
['ADMIN BUILDING', 'CANTEEN', 'CLASSROOM', 'CORE5 GF', 'DISANG ROAD', 'HOSTEL ROOM', 'LIBRARY', 'NEW SAC', 'OPEN ROAD', 'ROADS', 'VISITOR ROOM']


In [166]:
print(
    wifi_chatbot(
        "What will the wifi be like at LIBRARY at 9 PM with crowd 3?",
        []
    )
)


## 📡 Campus Wi-Fi Prediction

**Location:** LIBRARY  
**Time:** 21:00  
**Expected crowd:** 3

### 🟠 Connectivity Risk: MEDIUM

**Estimated probability of poor connectivity:** `72.1%`

### 📊 Network Evidence

- Median signal strength: **-63.5 dBm**
- Median download speed: **0.34 Mbps**
- Ping failure rate: **0.0%**


### 🧑‍🎓 Student Survey Evidence

The survey contains **7 responses** associated with the **LIBRARY** category.

### 💡 Assessment
The supplied conditions correspond to a **moderate predicted connectivity risk** based on the collected measurements.


In [167]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Predictions ONLY on unseen test data
y_pred = rf_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.7906976744186046
Precision: 0.6923076923076923
Recall   : 0.8181818181818182
F1 Score : 0.75

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.77      0.82        53
           1       0.69      0.82      0.75        33

    accuracy                           0.79        86
   macro avg       0.78      0.80      0.78        86
weighted avg       0.80      0.79      0.79        86


Confusion Matrix:
[[41 12]
 [ 6 27]]


In [168]:
from datetime import datetime
validation_history = []

In [169]:
def validate_live_prediction(
    location,
    time,
    crowd,
    actual_download,
    actual_ping,
    internet_working
):
    
    # -----------------------------
    # 1. MODEL PREDICTION
    # -----------------------------
    
    prediction = get_wifi_prediction(
        location=location,
        time=time,
        crowd=crowd
    )
    
    predicted_risk = prediction["risk"]
    predicted_probability = float(
        prediction["poor_connectivity_probability"]
    )
    
    # -----------------------------
    # 2. ACTUAL LIVE CONDITION
    # -----------------------------
    
    actual_download = float(actual_download)
    actual_ping = float(actual_ping)
    
    # Same rule used during training
    ping_failed = 0 if internet_working else 1
    
    actual_poor = (
        (actual_download <= DOWNLOAD_THRESHOLD) or
        (ping_failed == 1)
    )
    
    actual_risk = "HIGH" if actual_poor else "LOW"
    
    # -----------------------------
    # 3. COMPARE
    # -----------------------------
    
    predicted_poor = predicted_risk == "HIGH"
    
    matched = predicted_poor == actual_poor
    
    # -----------------------------
    # 4. SAVE VALIDATION
    # -----------------------------
    
    record = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Location": location,
        "time": time,
        "crowd": crowd,
        "predicted_risk": predicted_risk,
        "predicted_probability": predicted_probability,
        "actual_download": actual_download,
        "actual_ping": actual_ping,
        "actual_risk": actual_risk,
        "prediction_matched": matched
    }
    
    validation_history.append(record)
    
    # -----------------------------
    # 5. DISPLAY RESULT
    # -----------------------------
    
    result = f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        LIVE VALIDATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Location: {location}
Time: {time}
Expected Crowd: {crowd}

MODEL PREDICTION
Risk: {predicted_risk}
Probability of Poor Connectivity:
{predicted_probability:.1%}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ACTUAL MEASUREMENT

Download Speed: {actual_download:.2f} Mbps
Ping: {actual_ping:.0f} ms
Internet Working: {"YES" if internet_working else "NO"}

Actual Condition: {actual_risk}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━

VALIDATION RESULT

{"✓ PREDICTION MATCHED ACTUAL CONDITION"
 if matched
 else
 "✗ PREDICTION DID NOT MATCH ACTUAL CONDITION"}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""
    
    return result

In [170]:
def prediction_for_validation(location, time, crowd):

    result = get_wifi_prediction(
        location=location,
        time=time,
        crowd=crowd
    )

    return f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        MODEL PREDICTION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Location: {location}
Time: {time}
Expected Crowd: {crowd}

Risk: {result["risk"]}

Probability of Poor Connectivity:
{float(result["poor_connectivity_probability"]):.1%}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""


predict_button.click(
    prediction_for_validation,
    inputs=[
        validation_location,
        validation_time,
        validation_crowd
    ],
    outputs=prediction_output
)

NameError: name 'predict_button' is not defined

In [ ]:
demo.launch(share=True)

In [ ]:
with gr.Blocks() as demo:

    with gr.Tab("Live Validation"):

        gr.Markdown("""
        ## 📱 Live Wi-Fi Validation

        Generate the model prediction first.
        Then test the same Wi-Fi network on your phone
        and enter the actual measurements.
        """)

        validation_location = gr.Dropdown(
            choices=sorted(df["Location"].dropna().unique()),
            label="Location"
        )

        validation_time = gr.Textbox(
            label="Time",
            value="9:00 PM"
        )

        validation_crowd = gr.Slider(
            minimum=1,
            maximum=5,
            value=3,
            step=1,
            label="Expected Crowd"
        )

        predict_button = gr.Button(
            "🔮 Generate Prediction"
        )

        prediction_output = gr.Textbox(
            label="Model Prediction",
            lines=8
        )

        actual_download = gr.Number(
            label="Actual Download Speed (Mbps)"
        )

        actual_ping = gr.Number(
            label="Actual Ping (ms)"
        )

        internet_working = gr.Checkbox(
            label="Internet Working?",
            value=True
        )

        validate_button = gr.Button(
            "✅ Validate Against Live Measurement"
        )

        validation_output = gr.Textbox(
            label="Validation Result",
            lines=15
        )


    # =====================================
    # PREDICTION BUTTON
    # =====================================

    def prediction_for_validation(location, time, crowd):

        result = get_wifi_prediction(
            location=location,
            time=time,
            crowd=crowd
        )

        return f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        MODEL PREDICTION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Location: {location}
Time: {time}
Expected Crowd: {crowd}

Risk: {result["risk"]}

Probability of Poor Connectivity:
{float(result["poor_connectivity_probability"]):.1%}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""


    predict_button.click(
        prediction_for_validation,
        inputs=[
            validation_location,
            validation_time,
            validation_crowd
        ],
        outputs=prediction_output
    )


    # =====================================
    # LIVE VALIDATION BUTTON
    # =====================================

    def run_live_validation(
        location,
        time,
        crowd,
        actual_download,
        actual_ping,
        internet_working
    ):

        return validate_live_prediction(
            location=location,
            time=time,
            crowd=crowd,
            actual_download=actual_download,
            actual_ping=actual_ping,
            internet_working=internet_working
        )


    validate_button.click(
        run_live_validation,
        inputs=[
            validation_location,
            validation_time,
            validation_crowd,
            actual_download,
            actual_ping,
            internet_working
        ],
        outputs=validation_output
    )


# =====================================
# LAUNCH
# =====================================

demo.launch(share=True)

In [ ]:
def prediction_for_validation(location, time, crowd):

    result = get_wifi_prediction(
        location=location,
        time=time,
        crowd=crowd
    )

    return f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        MODEL PREDICTION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Location: {location}
Time: {time}
Expected Crowd: {crowd}

Risk: {result["risk"]}

Probability of Poor Connectivity:
{float(result["poor_connectivity_probability"]):.1%}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

In [ ]:
def run_live_validation(
    location,
    time,
    crowd,
    actual_download,
    actual_ping,
    internet_working
):

    return validate_live_prediction(
        location=location,
        time=time,
        crowd=crowd,
        actual_download=actual_download,
        actual_ping=actual_ping,
        internet_working=internet_working
    )

In [ ]:
validate_button.click(
    run_live_validation,
    inputs=[
        validation_location,
        validation_time,
        validation_crowd,
        actual_download,
        actual_ping,
        internet_working
    ],
    outputs=validation_output
)

In [ ]:
predict_button.click(
    prediction_for_validation,
    inputs=[
        validation_location,
        validation_time,
        validation_crowd
    ],
    outputs=prediction_output
)

In [ ]:
def run_live_validation(
    location,
    time,
    crowd,
    actual_download,
    actual_ping,
    internet_working
):

    return validate_live_prediction(
        location=location,
        time=time,
        crowd=crowd,
        actual_download=actual_download,
        actual_ping=actual_ping,
        internet_working=internet_working
    )

In [ ]:
validate_button.click(
    run_live_validation,
    inputs=[
        validation_location,
        validation_time,
        validation_crowd,
        actual_download,
        actual_ping,
        internet_working
    ],
    outputs=validation_output
)

In [ ]:
validation_df = pd.DataFrame(validation_history)

validation_accuracy = (
    validation_df["prediction_matched"].mean()
)

print(
    f"Live validation accuracy: "
    f"{validation_accuracy:.1%}"
)

In [ ]:
display(validation_df)

In [ ]:
print("Total live tests:", len(validation_df))

print(
    "Correct predictions:",
    validation_df["prediction_matched"].sum()
)

print(
    "Live validation accuracy:",
    f"{validation_df['prediction_matched'].mean():.1%}"
)

In [ ]:
demo.launch(share=True)